<!-- nav -->
[← 7 · From one language to another](07_From_One_Language_To_Another.ipynb) · [Home](../../README.md) · [Programs design →](../../docs/PROGRAMS.md)

# 8 · From rules to code

**The problem.** The cold chain's rule, "the temperature stays between 2 and 8 °C", is kept as data with
mbse-expressions, so it's stored, checked and translated in one place. But the nightly pipeline needs it as a Python
function, written the way a Python programmer would write it, in a module that passes code review.

A **bridge** carries expressions between an mbse-expressions dialect and a language's syntax trees, both ways. Rules
become code, generated rather than hand-copied, and existing code becomes rules. That closes the loop the mbse
repositories are for: a specification that is executable, and the source code generated from it. The bridge works in
TypeScript too, so a TypeScript program can generate Python, and read it.

In [1]:
import { Evaluators, Expressions as E } from "@mbse/expressions";
import * as Excel from "@mbse/expressions/Dialects/Excel";
import * as PythonDialect from "@mbse/expressions/Dialects/Python";
import * as Translators from "@mbse/expressions/Translators";
import { Python as Bridge } from "@mbse/programs/Bridges";
import { Python314, Syntax as P } from "@mbse/programs/Python";
import { Proxies, Schemas } from "@mbse/schemas/Framework";

const Py = P.LANGUAGE.Builders;
const Shipment = new Schemas.OfObject.Builder().name("Shipment").ref()
  .properties((p: any) => p.name("celsius").of((t: any) => t.as_native(Number)),
    (p: any) => p.name("hours").of((t: any) => t.as_native(BigInt))).create();
const store: any = new Proxies.OfStore();
store.register(Shipment);
const self = E.variable("this");
const celsius = self.celsius;
const rules: Record<string, any> = {
  in_range: E.literal(2.0).le(celsius).and_(celsius.le(8.0)).data,
  on_time: self.hours.le(72n).data,
};

## Step 1: a rule becomes a function

mbse-expressions translates the rule from its neutral form (Basic) to its Python dialect; the bridge turns that into a
Python syntax tree, here a function of `this` that returns it:

In [2]:
const toPython = Translators.between(E.DIALECT, PythonDialect.Expressions.DIALECT);
const inRange = Bridge.function_("in_range", ["this"], toPython.forward(rules.in_range));
console.log(Python314.print(inRange));

def in_range(this):
    return 2.0 <= this.celsius and this.celsius <= 8.0


## Step 2: a module of checks

Every rule becomes a function, and the functions a module, built as one tree and printed once:

In [3]:
const terms = Object.fromEntries(Object.entries(rules).map(([name, rule]) => [name, toPython.forward(rule)]));
const checks = Py.Module().body(Object.entries(terms).map(([name, term]) => Bridge.function_(name, ["this"], term))).create();
console.log(Python314.print(checks));

def in_range(this):
    return 2.0 <= this.celsius and this.celsius <= 8.0


def on_time(this):
    return this.hours <= 72



The module is ordinary Python. TypeScript can't run it, but mbse-expressions' model of Python's rules can evaluate
each generated function's expression, and it agrees with the rules themselves:

In [4]:
const shipments = [store.Shipment().celsius(5.5).hours(30n).create(), store.Shipment().celsius(9.1).hours(80n).create()];
for (const shipment of shipments) {
  const generated = Object.values(terms).map((term) => PythonDialect.Evaluators.OfAny(term, { this: shipment }));
  const evaluated = Object.values(rules).map((rule) => Evaluators.predicate(rule, shipment));
  console.log(generated, evaluated);
}

[ true, true ] [ true, true ]


[ false, false ] [ false, false ]


## Step 3: code becomes a rule

The bridge reads the other way too. A check already written in Python, in the pipeline nobody wants to rewrite, becomes
a rule: stored as data, evaluated by Basic's rules, translated to Excel for the QA team. Python's own parser exists only
in Python; through the bridge, TypeScript reads Python source as well.

In [5]:
const legacy = Python314.parse(`def fresh(this):
    return this.hours <= 48 and this.celsius >= 2.0
`) as P.Module;
const term = Bridge.term(((legacy.body[0] as P.FunctionDef).body[0] as P.Return).value!);
const rule = Translators.between(PythonDialect.Expressions.DIALECT, E.DIALECT).forward(term);
console.log(Evaluators.predicate(rule, shipments[0]));
console.log(Excel.Text.ToText(Translators.between(E.DIALECT, Excel.Expressions.DIALECT).forward(rule)));

true


=AND(this.hours <= 48, this.celsius >= 2.0)


## Step 4: what a rule can't hold

A rule is smaller than a program: pure, total, with no statements. Code outside that, here an identity test, can't
become one, and the bridge says where:

In [6]:
try {
  Bridge.term(((Python314.parse("this.celsius is None\n") as P.Module).body[0] as P.Expr).value);
} catch (error) {
  console.log((error as Error).message);
}

comparisons[0]: the operator 'is' has no counterpart in the Python dialect


## Why it works this way

- **Specifications generate code.** A rule written once, as data, is checked, stored and translated by
  mbse-expressions, and becomes production source here, printed by the language's own printer. Nobody copies it by
  hand, so nobody copies it wrong.
- **Both directions.** Existing code can be read into rules, so formalizing a system can start from what's already
  written and proceed step by step.
- **Trees all the way.** From the rule's terms to the dialect's terms to Python's syntax nodes, nothing pastes text, so
  precedence and quoting are right by construction.

That's the series: programs as data, changed safely, checked against standards, with names resolved, across languages,
shared between tools, translated between languages, and generated from the rules of an executable specification.

<!-- nav -->
[← 7 · From one language to another](07_From_One_Language_To_Another.ipynb) · [Home](../../README.md) · [Programs design →](../../docs/PROGRAMS.md)